In [3]:
# import scanpy as sc
# import pandas as pd
# adata =sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/scRNA.h5ad")
# adata.obs['cell_type']=adata.obs['celltype_final']
# print(adata.obs['cell_type'].value_counts())
# adata.write_h5ad("/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/scRNA.h5ad")
# adata_st = sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/Spatial.h5ad")
# st_label = adata_st.uns['density']
# st_label_df = pd.DataFrame(st_label)
# st_label_df.to_csv("/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/Data4/dataset4_density.csv")

In [4]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/Spatial.h5ad" \
    --n_epochs 150 \
    --output_dir ../deconv_results/DATA4 \
    --marker_selection_method l1 
#    --precomputed_marker_file "../deconv_results/STARmap/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4
   Batch size: 1024
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 128
   Loss type: MSE
   Lambda MMD: 0.05
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/scRNA.h5ad
   SC shape: (6948, 20007)
   SC avg counts/cell: 5354.0 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/Spatial.h5ad
   ST shape: (1000, 26160)
   Common genes: 17452
   SC final: (6948, 17452)
   ST final: (1000, 17452)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 46 clusters
Marker genes per cluster:
   0: 100 -> 100 (after L1)
   1: 100 -> 100 (after L1)
   10: 100 -> 100 (after L1)
   11: 100 -> 100 (after L1)
   12: 100 -> 100 (after L1)
   13:

VAE Training: 100%|██████████| 150/150 [01:56<00:00,  1.29epoch/s, Train=382.1084, Recon=375.7243, KL=63.8416, MMD=0.0108, Test=387.1717] 


📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (6948, 1080)
   Number of clusters: 46
   Computed embeddings shape: (6948, 128)
Computing cluster centers with mean aggregation...
   Completed: 46 clusters with mean centers and expressions
Extracting celltype-cluster mapping (using 'cell_type' column)...
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 7868 samples with 128 dims...
   UMAP visualization saved to: ../deconv_results/DATA4/modality_alignment_umap.png
Saving model to: ../deconv_results/DATA4/final_vae.pth
   Average cell counts: 5354.0 (for Stage 2 scale factor)
✅ Saved VAE model (weights only): ../deconv_results/DATA4/final_vae.pth
Saving cluster data to: ../deconv_results/DATA4/final_vae_cluster_data.npz
   ✓ Cluster IDs: 46
   ✓ Prototypes: (46, 128)
   ✓ Expressions (marker): (46, 1080)
   ✓ Expressions (full): 46 clusters × 17452 genes
   ✓ Celltype mapping: 46 clusters
✅ Sav

In [5]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset4/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/DATA4/final_vae.pth" \
    --output_dir "../deconv_results/DATA4/" \
    --n_epoch 250

Sample name: Spatial
Stage 1 model: ../deconv_results/DATA4/final_vae.pth
Output directory: ../deconv_results/DATA4/
Device: cpu
Weight threshold: 0.001
Loading pretrained VAE Encoder...
   VAE architecture: 1080 -> 128
   Output type: mse
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 6948 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/DATA4/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([46, 128])
   Cluster expressions (marker): torch.Size([46, 1080])
   Cluster expressions (all genes): 46 × 17452
   Loaded celltype mapping: 46 clusters
   Average cell counts: 5354.0
Loaded all genes list: 17452 genes
VAE Encoder loaded: 1080 -> 128
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '5', '6', '7', '8', '9']
Marker ge

GAT Training: 100%|██████████| 250/250 [09:31<00:00,  2.28s/epoch, Total=3.5686, MSE=0.3840, Spot_Cosine=0.1747, Gene_Cosine=0.3057, Pearson=0.2767, Gene_Pearson=0.5145, P_pat=3, M_pat=3, C_pat=15]  


Evaluating model results...
Applying weight threshold: 0.001
   Non-zero elements: 30000 -> 15370 (33.4%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/DATA4//Spatial_reconstructed_all_genes.csv
   Cell type composition...
   Using cluster→celltype mapping from Stage 1 checkpoint (46 entries).
   Found duplicate celltype names: ['EPCAM+ cells and cholangiocytes', 'NK,NKT and T cells', 'LSEC', 'Hepatocytes', 'B cells']. Merging corresponding cluster columns by summing weights.
   Columns before: 46, after merge: 6
   Saved cell composition (celltype): ../deconv_results/DATA4//Spatial_cell_composition.csv
   Saved cluster composition: ../deconv_results/DATA4//Spatial_cluster_composition.csv
   Using 1080 marker genes for reconstruction quality (consistent with training objective)
   Cosine similarities saved: ../deconv_results/DATA4//Spatial_spot_cosine_similarity.csv

Plotting reconstr